## Set Hyperparameters

In [1]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input/datasets/akshaikumarn")

train_root = INPUT_ROOT / "bytebot-train"
val_root = INPUT_ROOT / "bytebot-val"
tokenizer_root = INPUT_ROOT / "bytebot-tokenizer"

TRAIN_DIR = train_root / "train" if (train_root / "train").exists() else train_root
VAL_DIR = val_root / "val" if (val_root / "val").exists() else val_root

TOKENIZER_PATH = tokenizer_root / "tokenizer.json"
CKPT_DIR = Path("/kaggle/working/checkpoints")

VOCAB_SIZE = 34_048
D_MODEL = 768
N_LAYERS = 14
N_HEADS = 12
FFN_DIM = 2048
CONTEXT_LENGTH = 1024

MICRO_BATCH = 8
GRAD_ACCUM = 6
LEARNING_RATE = 3e-4
MIN_LR = 3e-5
WARMUP_FRACTION = 0.02
BETAS = (0.9, 0.95)
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
MAX_TRAIN_MINUTES = 600

LOG_EVERY = 250
EVAL_EVERY = 1000
EVAL_BATCHES = 20
CHECKPOINT_EVERY = 1000

NUM_WORKERS = 2
PREFETCH_FACTOR = 4
USE_COMPILE = True
SEED = 42

## Import Libraries

In [2]:
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

from accelerate import Accelerator, notebook_launcher
from accelerate.utils import set_seed, broadcast

print("torch", torch.__version__)

torch 2.11.0+cu128


## Data

In [3]:
train_files = sorted(TRAIN_DIR.glob("*.bin"))
val_files = sorted(VAL_DIR.glob("*.bin"))

assert train_files, f"no .bin files in {TRAIN_DIR}"
assert val_files, f"no .bin files in {VAL_DIR}"
assert TOKENIZER_PATH.exists(), f"missing {TOKENIZER_PATH}"

for name, files in [("train", train_files), ("val", val_files)]:
    size = sum(f.stat().st_size for f in files)
    print(f"{name}: {len(files)} files, {size / 1024**3:.2f} GB, {size / 2 / 1e9:.2f}B tokens")

train: 5 files, 7.08 GB, 3.80B tokens
val: 5 files, 0.04 GB, 0.02B tokens


## Random Sampling

In [4]:
class RandomTokenDataset(IterableDataset):
    def __init__(self, files, context_length, seed, rank=0):
        super().__init__()
        self.files = [str(f) for f in files]
        self.context_length = context_length
        self.seed = seed
        self.rank = rank
        self.maps = None

    def _open(self):
        maps = [np.memmap(f, dtype=np.uint16, mode="r") for f in self.files]
        maps = [m for m in maps if len(m) > self.context_length + 1]
        if not maps:
            raise ValueError("no token file is longer than the context length")
        lengths = np.array([len(m) for m in maps], dtype=np.float64)
        self.maps = maps
        self.weights = lengths / lengths.sum()

    def __iter__(self):
        if self.maps is None:
            self._open()

        info = torch.utils.data.get_worker_info()
        worker_id = info.id if info is not None else 0
        rng = np.random.default_rng([self.seed, self.rank, worker_id])
        L = self.context_length

        while True:
            data = self.maps[rng.choice(len(self.maps), p=self.weights)]
            start = rng.integers(0, len(data) - L)
            chunk = torch.from_numpy(data[start:start + L + 1].astype(np.int64))
            yield chunk[:-1], chunk[1:]

## Model

In [ ]:
from architecture import GPT

print("ByteBot architecture imported successfully!")

## Train

In [6]:
def train_worker():
    accelerator = Accelerator(mixed_precision="fp16", gradient_accumulation_steps=GRAD_ACCUM)
    device = accelerator.device
    rank = accelerator.process_index
    world = accelerator.num_processes
    main = accelerator.is_main_process

    set_seed(SEED)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)

    model = GPT(VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, FFN_DIM, CONTEXT_LENGTH).to(device)
    raw_model = model

    decay = [p for p in model.parameters() if p.ndim >= 2]
    no_decay = [p for p in model.parameters() if p.ndim < 2]  # norm gains
    optimizer = torch.optim.AdamW(
        [{"params": decay, "weight_decay": WEIGHT_DECAY},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=LEARNING_RATE, betas=BETAS, fused=True,
    )

    step, tokens_seen, elapsed_before, best_val = 0, 0, 0.0, float("inf")
    last_path = CKPT_DIR / "last.pt"
    if last_path.exists():
        ckpt = torch.load(last_path, map_location="cpu")
        raw_model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        step = ckpt["step"]
        tokens_seen = ckpt["tokens_seen"]
        elapsed_before = ckpt["elapsed_seconds"]
        best_val = ckpt["best_val_loss"]
        if main:
            print(f"resumed from step {step}, {tokens_seen / 1e9:.3f}B tokens")

    if USE_COMPILE:
        model = torch.compile(model)
    model, optimizer = accelerator.prepare(model, optimizer)

    train_ds = RandomTokenDataset(train_files, CONTEXT_LENGTH, seed=SEED + step, rank=rank)
    train_loader = DataLoader(
        train_ds, batch_size=MICRO_BATCH, num_workers=NUM_WORKERS,
        pin_memory=True, persistent_workers=True, prefetch_factor=PREFETCH_FACTOR,
    )

    val_iter = iter(RandomTokenDataset(val_files, CONTEXT_LENGTH, seed=SEED + 10**9, rank=rank))
    val_batches = []
    for _ in range(EVAL_BATCHES):
        pairs = [next(val_iter) for _ in range(MICRO_BATCH)]
        x = torch.stack([p[0] for p in pairs]).to(device)
        y = torch.stack([p[1] for p in pairs]).to(device)
        val_batches.append((x, y))

    total_seconds = MAX_TRAIN_MINUTES * 60
    warmup_seconds = total_seconds * WARMUP_FRACTION

    def get_lr(elapsed):
        if elapsed < warmup_seconds:
            return LEARNING_RATE * elapsed / warmup_seconds
        progress = min((elapsed - warmup_seconds) / (total_seconds - warmup_seconds), 1.0)
        return MIN_LR + 0.5 * (LEARNING_RATE - MIN_LR) * (1 + math.cos(math.pi * progress))

    def save(path, elapsed, full=True):
        if not main:
            return
        ckpt = {
            "model": raw_model.state_dict(),
            "config": dict(vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS,
                           n_heads=N_HEADS, ffn_dim=FFN_DIM, context_length=CONTEXT_LENGTH),
        }
        if full:
            ckpt.update(optimizer=optimizer.state_dict(), step=step, tokens_seen=tokens_seen,
                        elapsed_seconds=elapsed, best_val_loss=best_val)
        tmp = path.with_suffix(".tmp")
        torch.save(ckpt, tmp)
        os.replace(tmp, path)  # so a crash mid-save can't corrupt the old file
        print(f"saved {path.name} at step {step}")

    @torch.no_grad()
    def evaluate():
        model.eval()
        total = torch.zeros(1, device=device)
        for x, y in val_batches:
            with accelerator.autocast():
                _, loss = model(x, y)
            total += loss.float()
        model.train()
        return accelerator.gather(total / len(val_batches)).mean().item()

    if main:
        n_params = sum(p.numel() for p in raw_model.parameters())
        print(f"params: {n_params / 1e6:.1f}M, world size: {world}, "
              f"tokens/step: {MICRO_BATCH * GRAD_ACCUM * world * CONTEXT_LENGTH:,}")
        print("training started")

    model.train()
    train_iter = iter(train_loader)
    optimizer.zero_grad(set_to_none=True)

    start = time.monotonic()
    log_time, log_tokens = start, tokens_seen

    while True:
        t = torch.tensor([elapsed_before + time.monotonic() - start], dtype=torch.float64, device=device)
        elapsed = broadcast(t, from_process=0).item()
        if elapsed >= total_seconds:
            break

        lr = get_lr(elapsed)
        for group in optimizer.param_groups:
            group["lr"] = lr

        loss_sum = torch.zeros((), device=device)
        for _ in range(GRAD_ACCUM):
            x, y = next(train_iter)
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with accelerator.accumulate(model):
                with accelerator.autocast():
                    _, loss = model(x, y)
                accelerator.backward(loss)  # accelerate already divides by GRAD_ACCUM

            loss_sum += loss.detach().float()
            tokens_seen += x.numel() * world  # global count, all ranks

        assert accelerator.sync_gradients
        grad_norm = accelerator.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        step += 1

        if step % LOG_EVERY == 0:
            train_loss = accelerator.gather(loss_sum / GRAD_ACCUM).mean().item()
            if main:
                now = time.monotonic()
                tok_s = (tokens_seen - log_tokens) / (now - log_time)
                print(f"step {step:6d} | loss {train_loss:.4f} | gnorm {float(grad_norm):.2f} | "
                      f"lr {lr:.2e} | {tok_s / 1000:.1f}k tok/s | {elapsed / 60:.1f} min")
                log_time, log_tokens = now, tokens_seen

        if step % EVAL_EVERY == 0:
            val_loss = evaluate()
            if main:
                print(f"\nval | step {step} | loss {val_loss:.4f} | ppl {math.exp(min(val_loss, 20)):.2f}\n")
                if val_loss < best_val:
                    best_val = val_loss
                    save(CKPT_DIR / "best.pt", elapsed)

        if step % CHECKPOINT_EVERY == 0:
            save(last_path, elapsed)

    save(last_path, elapsed)
    accelerator.wait_for_everyone()

    if main:
        print(f"\ndone: {step:,} steps, {tokens_seen / 1e9:.3f}B tokens, {elapsed / 3600:.2f} h")

In [7]:
notebook_launcher(train_worker, num_processes=2)

Launching training on 2 CUDAs.
params: 125.3M, world size: 2, tokens/step: 98,304
training started

step    250 | loss 5.4636 | gnorm 0.68 | lr 3.00e-04 | 29.6k tok/s | 13.8 min
step    500 | loss 4.8647 | gnorm 0.56 | lr 3.00e-04 | 30.6k tok/s | 27.2 min
step    750 | loss 4.4762 | gnorm 0.50 | lr 2.98e-04 | 30.6k tok/s | 40.6 min
step   1000 | loss 4.2792 | gnorm 0.50 | lr 2.97e-04 | 30.6k tok/s | 54.0 min

val | step 1000 | loss 4.2026 | ppl 66.86

saved best.pt at step 1000
saved last.pt at step 1000
step   1250 | loss 4.0836 | gnorm 0.43 | lr 2.94e-04 | 29.8k tok/s | 67.7 min
step   1500 | loss 3.9580 | gnorm 0.42 | lr 2.91e-04 | 30.6k tok/s | 81.1 min
step   1750 | loss 3.8505 | gnorm 0.42 | lr 2.87e-04 | 30.6k tok/s | 94.5 min
step   2000 | loss 3.8038 | gnorm 0.40 | lr 2.83e-04 | 30.6k tok/s | 107.9 min

val | step 2000 | loss 3.7966 | ppl 44.55

saved best.pt at step 2000
saved last.pt at step 2000
step   2250 | loss 3.7051 | gnorm 0.37 | lr 2.78e-04 | 30.3k tok/s | 121.4 min
step   2500 | loss 3.7075 | gnorm 0.37 | lr 2.72e-04 | 30.6k tok/s | 134.8 min
step   2750 | loss 3.

## Retrain

In [ ]:
RETRAINING_LEARNING_RATE = 1e-4
RETRAINING_MIN_LR = 1e-5
RETRAINING_MINUTES = 660

source_checkpoint = Path("/kaggle/input/notebooks/akshaikumarn/bytebot-trainer/checkpoints/last.pt")

CKPT_DIR.mkdir(parents=True, exist_ok=True)
working_checkpoint = CKPT_DIR / "last.pt"

checkpoint = torch.load(
    source_checkpoint,
    map_location="cpu",
    weights_only=False
)

checkpoint["elapsed_seconds"] = 0.0

torch.save(checkpoint, working_checkpoint)
del checkpoint

LEARNING_RATE = RETRAINING_LEARNING_RATE
MIN_LR = RETRAINING_MIN_LR
MAX_TRAIN_MINUTES = RETRAINING_MINUTES

print("Checkpoint:", source_checkpoint)
print("Retraining learning rate:", RETRAINING_LEARNING_RATE)
print("Training limit:", RETRAINING_MINUTES, "minutes")

notebook_launcher(train_worker, num_processes=2)

Checkpoint: /kaggle/input/notebooks/akshaikumarn/bytebot-trainer/checkpoints/last.pt
Retraining learning rate: 0.0001
Training limit: 660 minutes
Launching training on 2 CUDAs.
resumed from step 11158, 1.097B tokens
params: 125.3M, world size: 2, tokens/step: 98,304
training started

step  11250 | loss 3.2594 | gnorm 0.40 | lr 3.62e-05 | 31.1k tok/s | 4.8 min
step  11500 | loss 3.2918 | gnorm 0.42 | lr 1.00e-04 | 32.6k tok/s | 17.3 min
step  11750 | loss 3.2347 | gnorm 0.43 | lr 9.99e-05 | 32.6k tok/s | 29.9 min
step  12000 | loss 3.1974 | gnorm 0.43 | lr 9.95e-05 | 32.6k tok/s | 42.5 min

val | step 12000 | loss 3.2764 | ppl 26.48

saved last.pt at step 12000
step  12250 | loss 3.2755 | gnorm 0.42 | lr 9.91e-05 | 31.9k tok/s | 55.3 min
step  12500 | loss 3.2763 | gnorm 0.42 | lr 9.84e-05 | 32.6k tok/s | 67.9 min
step  12750 | loss 3.3003 | gnorm 0.41 | lr 9.76e-05 | 32.6k tok/s | 80.4 min
step  13000 | loss 3.2586 | gnorm 0.43 | lr 9.67e-05 | 32.7k tok/s | 93.0 min

val | step 13000 | loss 3.2651 | ppl 26.18

saved last.pt at step 13000
step  13250 | loss 3.2470 | gnorm 0.44 | lr 9.55e-05 | 32.4k tok/s | 105.6 min
step  13500 | loss 3.2492 | gnorm 0.43 | lr 9.43e-05 | 32.4k tok/s | 118.3 min
step  13750 | loss 3.1686 | gnorm 0.44 | lr 9.28e-05 | 32.5k tok/s | 130.

In [ ]:
""" Due to Kernel Freeze (UI-to-Kernel Desynchronization, i.e., background process is still alive, but
the browser interface fails to render progress), the re-train output cell is frozen around 16.5k step.
"""

# Verify Process Completion

import torch

ckpt_path = "/kaggle/input/notebooks/akshaikumarn/bytebot-trainer/checkpoints/last.pt"

ckpt = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", ckpt.keys())

for key, value in ckpt.items():
    if not isinstance(value, dict):
        print(f"{key}: {value}")

Checkpoint keys: dict_keys(['model', 'config', 'optimizer', 'step', 'tokens_seen', 'elapsed_seconds', 'best_val_loss'])
step: 24142
tokens_seen: 2373255168
elapsed_seconds: 39602.797841765
best_val_loss: 3.1413516998291016
